# **VITS RASA 13**

In [ ]:
!pip install soundfile transformers torch huggingface_hub

In [ ]:
import json
import os
import torch
import soundfile as sf
from transformers import AutoModel, AutoTokenizer, AutoConfig

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Initializing AI4Bharat VITS Rasa 13 Model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "ai4bharat/vits_rasa_13"

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# ==========================================
# CONFIGURATION PATCH
# ==========================================
# Intercept the config and inject the missing attributes.
# We FORCE these to be integers because the custom tokenizer returns lists!
config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
config.pad_token_id = 0
config.bos_token_id = 1
config.eos_token_id = 2
# ==========================================

# 3. Load Model with the strictly integer-patched config
model = AutoModel.from_pretrained(model_id, config=config, trust_remote_code=True).to(device)

output_dir = "vits_rasa_hindi_eval(1)"
os.makedirs(output_dir, exist_ok=True)

# Configure Speaker IDs for Hindi (Mapped from model documentation)
speaker_id_female = 14 # Hindi Female
speaker_id_male = 1   # Hindi Male
style_id = 0           # Neutral/Default style

print("\nStarting Audio Generation...")

# Generation Loop
for gender, spk_id in [("Female", speaker_id_female), ("Male", speaker_id_male)]:
    print(f"\n--- Processing {gender} Voice (ID: {spk_id}) ---")

    for key, item in eval_data.items():
        text = item["text"]
        filename = f"{output_dir}/{item['id']}_{gender}.wav"

        print(f"Synthesizing {item['id']}...")

        # Tokenize the text
        inputs = tokenizer(text=text, return_tensors="pt").to(device)

        # Run inference with custom speaker and style IDs
        with torch.no_grad():
            outputs = model(
                inputs['input_ids'],
                speaker_id=spk_id,
                emotion_id=style_id
            )

        # Extract audio and save
        audio_data = outputs.waveform.squeeze().cpu().numpy()
        sample_rate = model.config.sampling_rate
        sf.write(filename, audio_data, sample_rate)

print(f"\n✅ All done! Evaluation audio files saved to '{output_dir}'.")

Loading evaluation dataset...
Initializing AI4Bharat VITS Rasa 13 Model...
Using device: cuda


Loading weights:   0%|          | 0/859 [00:00<?, ?it/s]


Starting Audio Generation...

--- Processing Female Voice (ID: 14) ---
Synthesizing HIN_01...
Synthesizing HIN_02...
Synthesizing HIN_03...
Synthesizing HIN_04...
Synthesizing HIN_05...
Synthesizing HIN_06...
Synthesizing HIN_07...
Synthesizing HIN_08...
Synthesizing HIN_09...
Synthesizing HIN_10...
Synthesizing HIN_11...
Synthesizing HIN_12...
Synthesizing HIN_13...
Synthesizing HIN_14...
Synthesizing HIN_15...
Synthesizing HIN_16...
Synthesizing HIN_17...
Synthesizing HIN_18...
Synthesizing HIN_19...
Synthesizing HIN_20...

--- Processing Male Voice (ID: 1) ---
Synthesizing HIN_01...
Synthesizing HIN_02...
Synthesizing HIN_03...
Synthesizing HIN_04...
Synthesizing HIN_05...
Synthesizing HIN_06...
Synthesizing HIN_07...
Synthesizing HIN_08...
Synthesizing HIN_09...
Synthesizing HIN_10...
Synthesizing HIN_11...
Synthesizing HIN_12...
Synthesizing HIN_13...
Synthesizing HIN_14...
Synthesizing HIN_15...
Synthesizing HIN_16...
Synthesizing HIN_17...
Synthesizing HIN_18...
Synthesizing HI

In [ ]:
import whisper
import jiwer
import json
import os
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Loading Whisper 'medium' model...")
model = whisper.load_model("medium")

output_dir = "vits_rasa_hindi_eval"
results = []

print("\nStarting Transcription and Evaluation...\n")

# Evaluation Loop
for gender in ["Female", "Male"]:
    for key, item in eval_data.items():
        file_id = item["id"]
        ground_truth = item["text"]
        audio_path = f"{output_dir}/{file_id}_{gender}.wav"

        if not os.path.exists(audio_path):
            continue

        # Transcribe
        transcription_result = model.transcribe(audio_path, language="hi")
        hypothesis = transcription_result["text"].strip()

        # Calculate Error Rates
        try:
            wer = jiwer.wer(ground_truth, hypothesis)
            cer = jiwer.cer(ground_truth, hypothesis)
        except ValueError:
            wer = 1.0
            cer = 1.0

        # Store data
        results.append({
            "Test_ID": file_id,
            "Category": key,
            "Gender": gender,
            "WER": round(wer, 3),
            "CER": round(cer, 3),
            "Original_Text": ground_truth,
            "Whisper_Transcription": hypothesis
        })

        print(f"[{file_id} - {gender}] WER: {wer:.3f} | CER: {cer:.3f}")

# Create DataFrame and summarize
df_results = pd.DataFrame(results)

print("\n" + "="*40)
print("🏆 VITS RASA EVALUATION SUMMARY")
print("="*40)

summary = df_results.groupby("Gender")[["WER", "CER"]].mean()
print(summary)

# Save report
csv_filename = "vits_rasa_whisper_evaluation.csv"
df_results.to_csv(csv_filename, index=False)
print(f"\n✅ Detailed breakdown saved to '{csv_filename}'.")

Loading evaluation dataset...
Loading Whisper 'medium' model...

Starting Transcription and Evaluation...

[HIN_01 - Female] WER: 0.600 | CER: 0.250
[HIN_02 - Female] WER: 0.667 | CER: 0.234
[HIN_03 - Female] WER: 0.625 | CER: 0.392
[HIN_04 - Female] WER: 0.364 | CER: 0.163
[HIN_05 - Female] WER: 0.538 | CER: 0.161
[HIN_06 - Female] WER: 0.833 | CER: 0.218
[HIN_07 - Female] WER: 0.750 | CER: 0.303
[HIN_08 - Female] WER: 0.357 | CER: 0.131
[HIN_09 - Female] WER: 0.571 | CER: 0.303
[HIN_10 - Female] WER: 0.429 | CER: 0.157
[HIN_11 - Female] WER: 0.214 | CER: 0.113
[HIN_12 - Female] WER: 0.692 | CER: 0.239
[HIN_13 - Female] WER: 0.615 | CER: 0.283
[HIN_14 - Female] WER: 0.857 | CER: 0.406
[HIN_15 - Female] WER: 0.533 | CER: 0.160
[HIN_16 - Female] WER: 0.412 | CER: 0.139
[HIN_17 - Female] WER: 0.500 | CER: 0.184
[HIN_18 - Female] WER: 0.714 | CER: 0.479
[HIN_19 - Female] WER: 0.667 | CER: 0.247
[HIN_20 - Female] WER: 0.733 | CER: 0.239
[HIN_01 - Male] WER: 0.400 | CER: 0.183
[HIN_02 - Mal

In [ ]:
import shutil
from google.colab import files

print("Zipping the audio files...")
# Compress the folder into a file named 'xtts_audio_results.zip'
shutil.make_archive('vits_rasa_audio_results', 'zip', 'vits_rasa_hindi_eval')

print("Starting download...")
# Trigger the browser download
files.download('vits_rasa_audio_results.zip')

Zipping the audio files...
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

print("Zipping the audio files...")
# Compress the folder into a file named 'xtts_audio_results.zip'
shutil.make_archive('vits_rasa_audio_results(1)', 'zip', 'vits_rasa_hindi_eval(1)')

print("Starting download...")
# Trigger the browser download
files.download('vits_rasa_audio_results(1).zip')

Zipping the audio files...
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>